# Cuaderno 12 — La base final: ETL, EDA y lo que falta

**Para qué sirve.** El proyecto tiene 189 archivos y 8,20 GB, con catorce versiones de
`base_modelado`, cuatro de `base_maestra` y cuatro paneles. Cualquiera que llegue de fuera no
sabría cuál es la base. Este cuaderno produce **una fuente de verdad por grano**, la somete a
pruebas que fallan ruidosamente, documenta cada columna, y mide tres cosas que deciden si la base
alcanza para lo que se quiere hacer con ella: cuánta información hay de verdad, cuánta etiqueta
falta, y qué habría que conseguir.

**Una decisión de diseño, declarada.** La base de procesos **no se reconstruye desde el JSON**. La
cadena de 5.070.819 releases a 2.736.387 procesos ya quedó verificada en el cuaderno 9, resta por
resta y sin residuo. Rehacerla aquí tomaría horas y solo arriesga romper algo que cuadra. Este
cuaderno parte de `base_maestra_v4` y pega los satélites, verificando en cada unión que no se
multipliquen filas.

**«Una sola base» no puede ser una sola tabla.** El trabajo tiene dos granos: una fila por
proceso y una fila por proveedor y año. Meterlos en una tabla exigiría o tirar el detalle del
proceso o repetir cada proveedor tantas veces como procesos tenga. Lo que sí se puede, y es lo
que hace este cuaderno, es que haya **un solo archivo autoritativo por grano** y que todo lo
demás quede declarado como histórico.

| Paso | Qué hace |
| --- | --- |
| 0 | Inventario: qué fuentes alimentan la base y qué archivos ya no alimentan nada |
| 1 | ETL: la base final de procesos, con las uniones verificadas |
| 2 | Pruebas de integridad; las críticas detienen el cuaderno |
| 3 | Diccionario de las columnas, con elegibilidad para el modelo y el motivo de exclusión |
| 4 | Redundancia: cuántos bloques de información independiente hay de verdad |
| 5 | Deriva temporal: cobertura por año y saltos estructurales |
| 6 | Universo: qué variables existen en cada clase de procedimiento |
| 7 | Cuánta etiqueta falta: la curva de precisión contra número de conglomerados |
| 8 | Modelo de referencia: ¿el conjunto reducido pierde algo? |
| 9 | Las brechas, medidas y con su fuente |
| 10 | Escribir el panel final y la lista de lo histórico |


## Paso 0 · Entorno e inventario

Primero, qué hay. Las siete fuentes que alimentan la base final quedan declaradas en un
diccionario explícito, y el manifiesto del cuaderno 9 sirve para clasificar los 189 archivos del
proyecto en canónicos, derivados e históricos. Lo que sale de aquí es la lista de candidatos a
`09_historico/`: archivos que ya no alimentan nada y solo confunden.

Si una fuente falta, el cuaderno sigue y lo declara en lugar de caerse. Eso importa porque el
cuaderno tiene que poder correr en una máquina que no sea la tuya.


In [ ]:
# ============ Paso 0 · Entorno e inventario de fuentes ============
import os, sys, json, time, io, contextlib, hashlib, datetime, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
os.chdir(r'C:\\SERCOP_FINAL')
R = '03_resultados'
OUT, OUT9, OUTF = f'{R}/c8_', f'{R}/c9_', f'{R}/final_'
SEMILLA = 20261001
con = duckdb.connect(); con.sql('SET preserve_insertion_order=false')

def seguro(f):
    """Ejecuta y deja la transaccion limpia: un error atrapado en DuckDB envenena las celdas
    siguientes si no se hace rollback."""
    try:
        return f()
    except Exception as e:
        try: con.rollback()
        except Exception: pass
        print('  error:', type(e).__name__, str(e)[:120]); return None

def filas(ruta):
    return seguro(lambda: con.sql(f"SELECT count(*) FROM read_parquet('{ruta}')").fetchone()[0])

def cols(ruta):
    r = seguro(lambda: con.sql(f"DESCRIBE SELECT * FROM read_parquet('{ruta}')").fetchall())
    return [x[0] for x in r] if r else []

# --- Las fuentes que alimentan la base final, declaradas ---
# La base de procesos NO se reconstruye desde el JSON: la cadena de 5.070.819 releases a
# 2.736.387 procesos ya quedo verificada en el cuaderno 9 y rehacerla aqui solo arriesga
# romper algo que cuadra. Este cuaderno parte de base_maestra_v4 y agrega los satelites.
FUENTES = {
    'base':        f'{R}/base_maestra_v4.parquet',      # tronco verificado, grano = ocid
    'etiqueta':    f'{OUT}etiqueta.parquet',            # y_fantasma por proceso
    'procedim':    f'{OUT9}procedimiento.parquet',      # clase y familia normalizadas
    'cri':         f'{OUT}cri_proceso.parquet',         # banderas rojas por proceso
    'panel':       f'{OUT}panel.parquet',               # grano = proveedor x anio
    'supercias':   f'{OUT}supercias.parquet',           # directorio societario
    'indices':     f'{R}/indices_red.parquet',          # idx_intermediacion e idx_cartel
}

print('='*74); print('FUENTES'); print('='*74)
inv = []
for k, ruta in FUENTES.items():
    existe = os.path.exists(ruta)
    inv.append((k, os.path.basename(ruta), existe,
                filas(ruta) if existe else None,
                len(cols(ruta)) if existe else 0,
                round(os.path.getsize(ruta)/1e6, 1) if existe else 0.0))
INV = pd.DataFrame(inv, columns=['rol','archivo','existe','filas','columnas','mb'])
print(INV.to_string(index=False))
faltan = INV[~INV.existe].rol.tolist()
if faltan:
    print(f'\nAVISO: faltan {faltan}. El cuaderno sigue y declara que no pudo unirlos.')

# --- Inventario completo del proyecto, desde el manifiesto del cuaderno 9 ---
print('\n' + '='*74); print('INVENTARIO DEL PROYECTO'); print('='*74)
MAN = None
try:
    MAN = pd.read_csv(f'{OUT9}manifiesto_archivos.csv')
    canon = set(os.path.basename(v) for v in FUENTES.values())
    def clasifica(a):
        b = os.path.basename(str(a))
        if b in canon: return '1 canonico'
        if any(t in b for t in ('base_modelado', 'base_maestra_v1', 'base_maestra_v2',
                                'base_maestra_v3', 'contaminad', '.tmp.')): return '3 historico'
        if b.startswith(('c8_', 'c9_', 'final_')): return '2 derivado'
        return '2 derivado'
    MAN['clase_archivo'] = MAN.archivo.map(clasifica)
    t = MAN.groupby('clase_archivo').agg(archivos=('archivo','size'),
                                         gb=('mb', lambda s: round(s.sum()/1024, 2)))
    print(t.to_string())
    hist = MAN[MAN.clase_archivo == '3 historico']
    print(f'\nCandidatos a 09_historico/: {len(hist)} archivos, '
          f'{hist.mb.sum()/1024:.2f} GB')
    if len(hist):
        print(hist[['archivo','mb','filas']].head(20).to_string(index=False))
except FileNotFoundError:
    print('No hay manifiesto: corre el paso 1 del cuaderno 9 para tenerlo.')

print('\nListo. El paso 1 arma la base final de procesos.')

## Paso 1 · ETL: la base final de procesos

Cada satélite se une con `LEFT JOIN` desde el tronco, y antes de unir se verifica que la llave sea
**única en el lado derecho**. Esta es la comprobación que más vale del paso: una unión contra una
tabla con llaves repetidas multiplica filas en silencio, no da ningún aviso, y todo lo que se
calcule después queda inflado sin que nadie lo note. Es el mismo tipo de falla que F13 y F14 del
cuaderno 10, que es por lo que aquí es una condición y no un comentario.

Al final se compara el número de filas contra el tronco. Si cambió, el cuaderno lo dice en
mayúsculas.


In [ ]:
# ============ Paso 1 · ETL: la base final de procesos ============
# Una sola tabla, grano = ocid, con todo lo que se usa pegado. Cada union es LEFT desde el
# tronco y se verifica que no multiplique filas: una union que duplica es el error mas caro
# de un ETL porque no da ningun aviso, solo infla los conteos.

N0 = filas(FUENTES['base'])
print(f'tronco base_maestra_v4: {N0:,} procesos')

piezas = ["SELECT b.* FROM read_parquet('" + FUENTES['base'] + "') b"]
sel, joins, notas = ['b.*'], [], []

def union(rol, llave, columnas, alias):
    """Agrega una union LEFT si el archivo existe y la llave es unica en el lado derecho."""
    if not os.path.exists(FUENTES.get(rol, '')):
        notas.append((rol, 'ausente', 0, 0)); return
    ruta, cs = FUENTES[rol], cols(FUENTES[rol])
    hay = [c for c in columnas if c in cs]
    if llave not in cs or not hay:
        notas.append((rol, f'sin llave {llave} o sin columnas', 0, 0)); return
    n  = filas(ruta)
    nd = seguro(lambda: con.sql(f"SELECT count(DISTINCT {llave}) FROM read_parquet('{ruta}')").fetchone()[0])
    if nd != n:
        notas.append((rol, f'llave repetida: {n:,} filas, {nd:,} llaves distintas', n, nd)); return
    joins.append(f"LEFT JOIN read_parquet('{ruta}') {alias} ON b.{llave} = {alias}.{llave}")
    sel.extend(f'{alias}.{c}' for c in hay)
    notas.append((rol, 'unido', n, len(hay)))

union('etiqueta', 'ocid', ['y_fantasma', 'en_catastro', 'f_marca', 'f_reactivacion'], 'e')
union('procedim', 'ocid', ['familia_procedimiento', 'clase', 'es_catalogo',
                           'universo_competencia'], 'p')
union('cri',      'ocid', [c for c in cols(FUENTES.get('cri', ''))
                           if c.startswith(('c_', 'k_', 'cri'))], 'k')

print('\nuniones:')
print(pd.DataFrame(notas, columns=['fuente','estado','filas_derecha','columnas_o_llaves']).to_string(index=False))

q = f"SELECT {', '.join(sel)} FROM read_parquet('{FUENTES['base']}') b " + ' '.join(joins)
seguro(lambda: con.sql(f'CREATE OR REPLACE VIEW fin AS {q}'))
N1 = filas_vista = seguro(lambda: con.sql('SELECT count(*) FROM fin').fetchone()[0])
NC = len(cols_fin := [r[0] for r in con.sql('DESCRIBE SELECT * FROM fin').fetchall()])

print(f'\nbase final: {N1:,} filas x {NC} columnas')
if N1 != N0:
    print(f'  ALTO: el tronco tenia {N0:,} y la union dejo {N1:,}. '
          f'{"Alguna union multiplico filas." if N1 > N0 else "Se perdieron filas."}')
else:
    print('  las uniones no cambiaron el numero de filas: correcto')

# columnas repetidas por sufijo de DuckDB (_1, _2) = union que trajo una columna ya existente
dup = [c for c in cols_fin if c[-2:-1] == '_' and c[-1].isdigit()]
print(f'columnas duplicadas por la union: {dup if dup else "ninguna"}')

con.sql(f"COPY fin TO '{OUTF}procesos.tmp.parquet' (FORMAT parquet, COMPRESSION zstd)")
os.replace(f'{OUTF}procesos.tmp.parquet', f'{OUTF}procesos.parquet')
print(f'\n  -> {OUTF}procesos.parquet  ({os.path.getsize(f"{OUTF}procesos.parquet")/1e6:.0f} MB)')
con.sql(f"CREATE OR REPLACE VIEW fin AS SELECT * FROM read_parquet('{OUTF}procesos.parquet')")

## Paso 2 · Pruebas de integridad

Diecisiete pruebas, cada una con su expectativa explícita. Las marcadas como **críticas** lanzan
una excepción: si el grano está roto o la etiqueta no se unió, lo que venga después es ruido y es
mejor que el cuaderno se detenga que producir una tabla bonita y falsa.

Las pruebas cubren tres cosas: el grano (una fila por `ocid`, sin nulos ni repetidos), las reglas
del RUC y el monto que ya estaban en el cuaderno 9, y las que salieron de las fallas de octubre
—que `n_oferentes` nulo esté solo en el catálogo, que ningún `n_oferentes` valga cero, que la
etiqueta tenga positivos—.

Una nota sobre `monto_tender`: si aparece en la base final la prueba lo marca para revisar, no
para fallar. Puede quedarse en la base como dato descriptivo; lo que no puede es entrar al modelo.


In [ ]:
# ============ Paso 2 · Pruebas de integridad ============
# Cada prueba es una consulta que devuelve un numero y una expectativa. Las criticas detienen
# el cuaderno: si el grano esta roto o la cadena no cuadra, todo lo que viene despues es ruido.

PRUEBAS = []
def prueba(nombre, sql, espera, critica=False, nota=''):
    v = seguro(lambda: con.sql(sql).fetchone()[0])
    ok = (v == espera) if not callable(espera) else bool(espera(v))
    PRUEBAS.append((nombre, v, espera if not callable(espera) else 'regla', ok, critica, nota))

cf = set(cols(f'{OUTF}procesos.parquet'))
def hay(*c): return all(x in cf for x in c)

# --- grano y unicidad ---
prueba('filas de la base final', 'SELECT count(*) FROM fin', N0, critica=True)
prueba('ocid distintos', 'SELECT count(DISTINCT ocid) FROM fin', N0, critica=True,
       nota='una fila por proceso')
prueba('ocid nulos', 'SELECT count(*) FROM fin WHERE ocid IS NULL', 0, critica=True)

# --- la cadena documentada en el cuaderno 9 ---
if hay('proveedor_ruc'):
    prueba('procesos con RUC de proveedor valido',
           "SELECT count(*) FROM fin WHERE TRIM(COALESCE(proveedor_ruc,'')) <> ''",
           lambda v: v is not None and v > 0, nota='el resto se excluye del modelado')
    prueba('RUC de 13 caracteres entre los validos',
           "SELECT count(*) FROM fin WHERE TRIM(COALESCE(proveedor_ruc,'')) <> '' "
           "AND length(TRIM(proveedor_ruc)) <> 13", 0, critica=True)
    prueba('RUC con caracteres no numericos',
           "SELECT count(*) FROM fin WHERE TRIM(COALESCE(proveedor_ruc,'')) <> '' "
           "AND NOT regexp_matches(TRIM(proveedor_ruc), '^[0-9]+$')", 0, critica=True)
    prueba('RUC que no termina en 001',
           "SELECT count(*) FROM fin WHERE TRIM(COALESCE(proveedor_ruc,'')) <> '' "
           "AND right(TRIM(proveedor_ruc), 3) <> '001'",
           lambda v: v is not None, nota='informativo, no es regla dura')

# --- monto ---
if hay('monto_final'):
    prueba('monto_final negativo', 'SELECT count(*) FROM fin WHERE monto_final < 0', 0, critica=True)
    prueba('monto_final nulo', 'SELECT count(*) FROM fin WHERE monto_final IS NULL',
           lambda v: v is not None)
    prueba('monto_final en cero', 'SELECT count(*) FROM fin WHERE monto_final = 0',
           lambda v: v is not None, nota='catalogo: el precio lo fija el convenio')
if 'monto_tender' in cf:
    PRUEBAS.append(('monto_tender presente en la base final', 'si', 'no deberia', False, False,
                    'excluido del modelo por origen, ver F08; puede quedarse en la base'))

# --- anio ---
if hay('anio_proceso'):
    prueba('anio fuera de 2015-2026',
           'SELECT count(*) FROM fin WHERE anio_proceso < 2015 OR anio_proceso > 2026', 0, critica=True)
    prueba('anios distintos', 'SELECT count(DISTINCT anio_proceso) FROM fin', 12)

# --- etiqueta ---
if hay('y_fantasma'):
    prueba('procesos etiquetados como fantasma',
           'SELECT count(*) FROM fin WHERE y_fantasma = 1', lambda v: v is not None and v > 0,
           critica=True, nota='si sale 0 la union de etiqueta fallo')
    prueba('etiqueta nula', 'SELECT count(*) FROM fin WHERE y_fantasma IS NULL',
           lambda v: v is not None, nota='procesos sin RUC: no se pueden etiquetar')

# --- procedimiento ---
if hay('clase'):
    prueba('procesos sin clase de procedimiento',
           "SELECT count(*) FROM fin WHERE clase IS NULL OR TRIM(clase) = ''",
           lambda v: v is not None and v < N0 * 0.01, nota='cobertura esperada 100 %')
if hay('n_oferentes', 'es_catalogo'):
    prueba('n_oferentes nulo FUERA del catalogo',
           'SELECT count(*) FROM fin WHERE n_oferentes IS NULL AND NOT COALESCE(es_catalogo, false)',
           lambda v: v is not None and v < N0 * 0.01,
           nota='el catalogo no tiene oferentes: eso es estructural, no faltante')
    prueba('n_oferentes en cero', 'SELECT count(*) FROM fin WHERE n_oferentes = 0', 0,
           nota='un cero delataria imputacion por constante')

T = pd.DataFrame(PRUEBAS, columns=['prueba','valor','esperado','pasa','critica','nota'])
T['resultado'] = np.where(T.pasa, 'PASA', np.where(T.critica, 'FALLA CRITICA', 'revisar'))
print(T[['prueba','valor','resultado','nota']].to_string(index=False))
T.to_csv(f'{OUTF}pruebas.csv', index=False)

mal = T[(~T.pasa) & T.critica]
print(f'\n{int(T.pasa.sum())} de {len(T)} pruebas pasan | criticas falladas: {len(mal)}')
print(f'  -> {OUTF}pruebas.csv')
if len(mal):
    print('\nALTO. Estas fallas invalidan lo que venga despues:')
    print(mal[['prueba','valor','esperado']].to_string(index=False))
    raise AssertionError('pruebas criticas falladas: revisar antes de seguir')
print('\nIntegridad en orden. El paso 3 documenta las columnas.')

In [ ]:
# ============ Paso 6 · Permutacion del universo de competencia (solo objetivo A) ============
# Le da su propio nulo a la cifra 0,770. Hay que correr ANTES el paso 2 (crea `comp`).
N_PERM6, K_P = 99, 15

def conglomerados_placebo(tam, pos, r):
    base = tam.drop(index=[c for c in pos if c in tam.index])
    if len(base) == 0: return np.array([], dtype=int)
    libres = base.copy(); sel = []
    for c in pos:
        if len(libres) == 0: libres = base.copy()
        n = int(tam[c]); v = libres.values
        cand = libres.index[v == n]
        if len(cand) == 0:
            dd = np.abs(v - n); cand = libres.index[dd == dd.min()]
        e = r.choice(np.asarray(cand)); sel.append(e)
        if e in libres.index: libres = libres.drop(index=e)
    return np.asarray(sel)

try:
    cols = [c for c in VARS['A'] if c in comp.columns]
    y  = comp['y_fantasma'].values.astype(int)
    gi = pd.factorize(comp['proveedor_ruc'].values)[0]
    an = comp['anio_proceso'].values
    X  = comp[cols].astype(float).values
    gp = np.unique(gi[y == 1]); tam = pd.Series(gi).value_counts(); Xu = X[y == 0]
    print(f'A universo de competencia: {y.sum()} positivos en {len(gp)} conglomerados, {len(cols)} variables')

    t0 = time.time(); obs = estadistico(X, y, gi, an, None, K_P, SEMILLA, Xu=Xu)
    print(f'  observado {obs:.3f}  ({time.time()-t0:.0f} s)')
    print(f'  estimado para {N_PERM6} permutaciones: {(time.time()-t0)*N_PERM6/60:.0f} minutos')

    nul, t0 = [], time.time()
    for b in range(N_PERM6):
        r = np.random.default_rng(SEMILLA + 200000 + b)
        sel = conglomerados_placebo(tam, gp, r)
        yp = np.isin(gi, sel).astype(int)
        nul.append(estadistico(X, yp, gi, an, None, K_P, SEMILLA + 200000 + b,
                               Xu=X[(y == 0) & (yp == 0)]))
        if (b+1) % 10 == 0:
            print(f'    {b+1}/{N_PERM6}  ({time.time()-t0:.0f} s, '
                  f'faltan ~{(time.time()-t0)/(b+1)*(N_PERM6-b-1)/60:.0f} min)', end='\r')
    v = np.array([x for x in nul if np.isfinite(x)])
    p1c = (1 + int((v >= obs).sum())) / (len(v) + 1)
    T6 = pd.DataFrame([{'caso': 'A fantasma · universo de competencia', 'observado': round(obs, 3),
        'n_perm': len(v), 'nulo_mediana': round(float(np.median(v)), 3),
        'nulo_p2.5': round(float(np.percentile(v, 2.5)), 3),
        'nulo_p97.5': round(float(np.percentile(v, 97.5)), 3),
        'p_una_cola': round(p1c, 4), 'p_minimo_alcanzable': round(1/(len(v)+1), 4)}])
    print('\n' + T6.to_string(index=False))
    T6.to_csv(f'{OUT}permutacion_competencia.csv', index=False)
    print(f'\n  -> {OUT}permutacion_competencia.csv')
except NameError as e:
    print('falta correr antes el paso 2 (crea `comp`) o el paso 0:', e)

In [ ]:
# ========================================================================
# Paso 2b - Pruebas que cruzan los dos granos
# Va DESPUES del paso 2 (el de las 17 pruebas) y ANTES del paso 3 (el diccionario).
# Copia TODO este archivo y pegalo en una celda nueva del cuaderno 12.
# ========================================================================

# ============ Paso 2b · Pruebas que cruzan los dos granos ============
# Las del paso 2 miran una tabla a la vez. Estas son las que de verdad fallan cuando algo se
# rompe entre cuadernos, porque comparan la tabla de procesos contra el panel y contra las
# reglas que aprendimos en octubre. Cada una tiene detras una falla real del cuaderno 10.

P2 = []
def chk(nombre, valor, ok, critica=False, nota=''):
    P2.append((nombre, valor, 'PASA' if ok else ('FALLA CRITICA' if critica else 'revisar'),
               critica, nota))

PA2 = pd.read_parquet(f'{OUT}panel.parquet')
cf = set(cols(f'{OUTF}procesos.parquet'))
cp = set(PA2.columns)

# --- 1. El monto del panel tiene que ser la suma de sus procesos (cuaderno 9, paso 7) ---
if {'proveedor_ruc', 'anio_proceso', 'monto_final'} <= cf and 'monto_anio' in cp:
    PA2[['proveedor_ruc','anio_proceso','monto_anio']].to_parquet(f'{R}/_pan_tmp.parquet', index=False)
    v = seguro(lambda: con.sql(f"""
        SELECT count(*) FROM (
          SELECT p.proveedor_ruc, p.anio_proceso, p.monto_anio AS decl,
                 coalesce(sum(f.monto_final), 0) AS suma
          FROM read_parquet('{R}/_pan_tmp.parquet') p
          LEFT JOIN fin f ON f.proveedor_ruc = p.proveedor_ruc
                         AND f.anio_proceso  = p.anio_proceso
          GROUP BY 1, 2, 3)
        WHERE abs(coalesce(decl, 0) - suma) > 1.0""").fetchone()[0])
    chk('monto_anio del panel = suma de monto_final de sus procesos', v, v == 0, critica=True,
        nota='si falla, el panel y los procesos describen universos distintos')
    os.remove(f'{R}/_pan_tmp.parquet')

# --- 2. La etiqueta tiene que ser la misma en los dos granos ---
if 'y_fantasma' in cf and {'proveedor_ruc', 'y_fantasma'} <= cp:
    pos_pan = set(PA2.loc[PA2.y_fantasma == 1, 'proveedor_ruc'].astype(str))
    pos_pro = seguro(lambda: set(con.sql(
        'SELECT DISTINCT proveedor_ruc FROM fin WHERE y_fantasma = 1').df().proveedor_ruc.astype(str)))
    if pos_pro is not None:
        chk('RUC positivos: panel contra procesos',
            f'{len(pos_pan)} vs {len(pos_pro)}', pos_pan == pos_pro, critica=True,
            nota='la etiqueta se construye una vez y se propaga; dos listas distintas es un error')
        if pos_pan != pos_pro:
            print('   solo en el panel:', sorted(pos_pan - pos_pro)[:5])
            print('   solo en procesos:', sorted(pos_pro - pos_pan)[:5])

# --- 3. La regla de F13: proveedor_unico existe solo donde hay proceso de ofertas ---
if {'proveedor_unico', 'es_catalogo'} <= cf:
    v = seguro(lambda: con.sql('SELECT count(*) FROM fin WHERE es_catalogo '
                               'AND proveedor_unico IS NOT NULL').fetchone()[0])
    chk('proveedor_unico con valor DENTRO del catalogo', v, v == 0,
        nota='el catalogo no tiene ofertas: un valor ahi es una imputacion escondida (F13)')
    v = seguro(lambda: con.sql('SELECT count(*) FROM fin WHERE NOT es_catalogo '
                               'AND proveedor_unico IS NULL').fetchone()[0])
    chk('proveedor_unico nulo FUERA del catalogo', v, v is not None and v < N0 * 0.001,
        nota='unos pocos son normales; muchos serian un hueco de cobertura')

# --- 4. La ventana expansiva: el primer anio no puede tener historial ---
# Se prueba SOLO sobre el conjunto declarado como rezagado. Que una variable de red no
# declarada ahi traiga valor en el primer anio no es un error: puede ser del anio en curso
# a proposito. La regla dura aplica a las que la tesis llama anticipadas.
try:
    with open(f'{OUT}vars.json', encoding='utf-8') as f: _V = json.load(f)
    vred = [c for c in _V.get('A_anticipada', []) if c in cp]
    fuente_vred = 'conjunto A_anticipada'
except FileNotFoundError:
    vred, fuente_vred = [], 'sin vars.json'
if not vred:
    vred = [c for c in cp if c.startswith(('red_', 'cob_', 'com_'))]
    fuente_vred = 'prefijos red_/cob_/com_ (aproximacion)'
print(f'ventana expansiva: se prueban {len(vred)} variables del {fuente_vred}')
if vred and 'anio_proceso' in cp:
    a0 = int(PA2.anio_proceso.min())
    s0 = PA2[PA2.anio_proceso == a0]
    llenas = []
    for c in vred:
        x = pd.to_numeric(s0[c], errors='coerce')
        if len(x) and (x.fillna(0) != 0).mean() > 0.005: llenas.append(c)
    chk(f'variables rezagadas con valor en el primer anio ({a0})', len(llenas), len(llenas) == 0,
        critica=fuente_vred.startswith('conjunto'),
        nota='si alguna trae valor, la ventana incluye el anio en curso y no es rezagada')
    if llenas: print('   con valor en el primer anio:', llenas[:10])

# --- 5. Columnas muertas y columnas gemelas exactas ---
MU = 60000
sub = PA2.sample(min(MU, len(PA2)), random_state=SEMILLA)
muertas = [c for c in PA2.columns if sub[c].isna().all()]
chk('columnas totalmente nulas en el panel', len(muertas), len(muertas) == 0,
    nota='una columna sin un solo valor no deberia viajar en la base')
if muertas: print('   muertas:', muertas[:15])

firmas = {}
gemelas = []
for c in sub.columns:
    try:
        h = hashlib.md5(pd.util.hash_pandas_object(sub[c], index=False).values.tobytes()).hexdigest()
    except Exception:
        continue
    if h in firmas: gemelas.append((firmas[h], c))
    else: firmas[h] = c
chk('pares de columnas con contenido identico', len(gemelas), len(gemelas) == 0,
    nota='identicas valor por valor: una de las dos es un sobrante')
if gemelas:
    print('   identicas:')
    for a, b in gemelas[:12]: print(f'     {a}  ==  {b}')

# --- 6. La especificacion congelada del cuaderno 11 sigue intacta ---
try:
    with open(f'{OUT}especificacion_congelada.json', encoding='utf-8') as f: esp = json.load(f)
    with open(f'{OUT}vars.json', encoding='utf-8') as f: V2 = json.load(f)
    cambios = []
    for k, d in esp.get('conjuntos', {}).items():
        if k not in V2: cambios.append(f'{k}: ya no existe'); continue
        fr = hashlib.sha256('\n'.join(sorted(V2[k])).encode('utf-8')).hexdigest()[:16]
        if fr != d.get('firma'): cambios.append(f'{k}: {d.get("firma")} -> {fr}')
    chk(f'firmas de la especificacion congelada ({esp.get("sello","?")})',
        len(cambios), len(cambios) == 0,
        nota='si cambio una firma, el conjunto se toco y hay que anotarlo en el cuaderno 10')
    if cambios:
        print('   conjuntos que cambiaron desde el congelamiento:')
        for c in cambios: print('     ', c)
except FileNotFoundError:
    chk('especificacion congelada', 'no existe', True,
        nota='corre la pieza 1 del cuaderno 11 para generarla')

T2 = pd.DataFrame(P2, columns=['prueba','valor','resultado','critica','nota'])
print(T2[['prueba','valor','resultado','nota']].to_string(index=False))
T2.to_csv(f'{OUTF}pruebas_cruzadas.csv', index=False)
mal2 = T2[(T2.resultado == 'FALLA CRITICA')]
print(f'\n{int((T2.resultado == "PASA").sum())} de {len(T2)} pasan | criticas falladas: {len(mal2)}')
print(f'  -> {OUTF}pruebas_cruzadas.csv')
if len(mal2):
    print('\nALTO:'); print(mal2[['prueba','valor','nota']].to_string(index=False))
    raise AssertionError('pruebas cruzadas criticas falladas')

In [ ]:
# ============ Paso 6 · Permutacion del universo de competencia (solo objetivo A) ============
# Le da su propio nulo a la cifra 0,770. Hay que correr ANTES el paso 2 (crea `comp`).
N_PERM6, K_P = 99, 15

def conglomerados_placebo(tam, pos, r):
    base = tam.drop(index=[c for c in pos if c in tam.index])
    if len(base) == 0: return np.array([], dtype=int)
    libres = base.copy(); sel = []
    for c in pos:
        if len(libres) == 0: libres = base.copy()
        n = int(tam[c]); v = libres.values
        cand = libres.index[v == n]
        if len(cand) == 0:
            dd = np.abs(v - n); cand = libres.index[dd == dd.min()]
        e = r.choice(np.asarray(cand)); sel.append(e)
        if e in libres.index: libres = libres.drop(index=e)
    return np.asarray(sel)

try:
    cols = [c for c in VARS['A'] if c in comp.columns]
    y  = comp['y_fantasma'].values.astype(int)
    gi = pd.factorize(comp['proveedor_ruc'].values)[0]
    an = comp['anio_proceso'].values
    X  = comp[cols].astype(float).values
    gp = np.unique(gi[y == 1]); tam = pd.Series(gi).value_counts(); Xu = X[y == 0]
    print(f'A universo de competencia: {y.sum()} positivos en {len(gp)} conglomerados, {len(cols)} variables')

    t0 = time.time(); obs = estadistico(X, y, gi, an, None, K_P, SEMILLA, Xu=Xu)
    print(f'  observado {obs:.3f}  ({time.time()-t0:.0f} s)')
    print(f'  estimado para {N_PERM6} permutaciones: {(time.time()-t0)*N_PERM6/60:.0f} minutos')

    nul, t0 = [], time.time()
    for b in range(N_PERM6):
        r = np.random.default_rng(SEMILLA + 200000 + b)
        sel = conglomerados_placebo(tam, gp, r)
        yp = np.isin(gi, sel).astype(int)
        nul.append(estadistico(X, yp, gi, an, None, K_P, SEMILLA + 200000 + b,
                               Xu=X[(y == 0) & (yp == 0)]))
        if (b+1) % 10 == 0:
            print(f'    {b+1}/{N_PERM6}  ({time.time()-t0:.0f} s, '
                  f'faltan ~{(time.time()-t0)/(b+1)*(N_PERM6-b-1)/60:.0f} min)', end='\r')
    v = np.array([x for x in nul if np.isfinite(x)])
    p1c = (1 + int((v >= obs).sum())) / (len(v) + 1)
    T6 = pd.DataFrame([{'caso': 'A fantasma · universo de competencia', 'observado': round(obs, 3),
        'n_perm': len(v), 'nulo_mediana': round(float(np.median(v)), 3),
        'nulo_p2.5': round(float(np.percentile(v, 2.5)), 3),
        'nulo_p97.5': round(float(np.percentile(v, 97.5)), 3),
        'p_una_cola': round(p1c, 4), 'p_minimo_alcanzable': round(1/(len(v)+1), 4)}])
    print('\n' + T6.to_string(index=False))
    T6.to_csv(f'{OUT}permutacion_competencia.csv', index=False)
    print(f'\n  -> {OUT}permutacion_competencia.csv')
except NameError as e:
    print('falta correr antes el paso 2 (crea `comp`) o el paso 0:', e)

## Paso 3 · Diccionario de datos

Era el pendiente más citado del cuaderno 9. Cada columna con su tipo, cobertura, cardinalidad y
familia, y sobre todo dos columnas que convierten un listado en un documento de tesis:
**`elegible`** y **`por_que_no`**.

Los motivos de exclusión son cuatro y cada uno corresponde a una decisión ya registrada: es llave
o fecha; es la etiqueta o deriva de ella; está excluida por origen (`monto_tender`, las derivadas
del SRI); o es no numérica y sirve para describir pero no para modelar. Un tribunal puede recorrer
esta tabla y preguntar por cualquier fila.


In [ ]:
# ============ Paso 3 · Diccionario de datos ============
# Lo que faltaba del cuaderno 9: cada columna con su tipo, cobertura, cardinalidad, familia,
# y sobre todo si es ELEGIBLE como variable del modelo y, si no, por que no. Esa ultima
# columna es la que convierte un diccionario en un documento util para la tesis.

FAMILIA = {'red_': 'red bipartita entidad-proveedor', 'cob_': 'red de co-oferta (rivales)',
           'com_': 'comunidades de la red', 'ent_': 'entidad contratante',
           'rep_': 'representante legal', 'sup_': 'directorio societario',
           'idx_': 'indice teorico', 'c_': 'bandera roja (CRI)', 'k_': 'bandera roja (CRI)',
           'cri': 'indice CRI', 'monto': 'monto', 'proveedor_': 'proveedor',
           'entidad_': 'entidad', 'f_': 'fecha de la marca del SRI'}
def familia(c):
    for p, d in FAMILIA.items():
        if c.startswith(p): return d
    return 'base del proceso'

LLAVES    = {'ocid', 'proveedor_ruc', 'entidad_ruc', 'anio_proceso', 'fecha_release', 'fecha_proc'}
ETIQUETAS = {'y_fantasma', 'y_intragrupo', 'en_catastro', 'f_marca', 'f_reactivacion',
             'proveedor_es_fantasma', 'entidad_es_fantasma', 'proveedor_fantasma_estado',
             'proveedor_fantasma_detalle'}
POR_ORIGEN = {'monto_tender': 'su patron de nulos ES el tipo de procedimiento (F08)'}

VARS = {}
try:
    with open(f'{OUT}vars.json', encoding='utf-8') as f: VARS = json.load(f)
    EN_MODELO = set(VARS.get('A', [])) | set(VARS.get('B', []))
    print(f'conjuntos del modelo leidos: { {k: len(v) for k, v in VARS.items()} }')
except FileNotFoundError:
    EN_MODELO = set(); print('no hay c8_vars.json: la columna de elegibilidad queda incompleta')

perfil = seguro(lambda: con.sql("""
SELECT * FROM (SUMMARIZE SELECT * FROM fin)""").df())
if perfil is None:
    print('SUMMARIZE fallo; se arma el diccionario con pandas por bloques')

tipos = {r[0]: r[1] for r in con.sql('DESCRIBE SELECT * FROM fin').fetchall()}
dic = []
for c, t in tipos.items():
    fila = {'columna': c, 'tipo': t, 'familia': familia(c)}
    if perfil is not None and 'column_name' in perfil.columns:
        r = perfil[perfil.column_name == c]
        if len(r):
            r = r.iloc[0]
            fila['pct_nulo']   = round(float(r.get('null_percentage') or 0), 2)
            fila['distintos']  = r.get('approx_unique')
            fila['min']        = str(r.get('min'))[:22]
            fila['max']        = str(r.get('max'))[:22]
    # elegibilidad
    if c in LLAVES:            el, por = 'no', 'llave o fecha'
    elif c in ETIQUETAS:       el, por = 'no', 'es la etiqueta o deriva de ella'
    elif c in POR_ORIGEN:      el, por = 'no', POR_ORIGEN[c]
    elif c in EN_MODELO:       el, por = 'si', ''
    elif t.upper() in ('VARCHAR', 'BOOLEAN', 'DATE', 'TIMESTAMP'):
                               el, por = 'no', 'no numerica: sirve para describir, no para el modelo'
    else:                      el, por = 'no', 'numerica pero fuera de los conjuntos A y B'
    fila['elegible'], fila['por_que_no'] = el, por
    dic.append(fila)

D = pd.DataFrame(dic)
D.to_csv(f'{OUTF}diccionario.csv', index=False)
print(f'\ndiccionario: {len(D)} columnas  ->  {OUTF}diccionario.csv\n')

print('Por familia:')
print(D.groupby('familia').agg(columnas=('columna','size'),
        elegibles=('elegible', lambda s: int((s == 'si').sum())),
        nulo_medio=('pct_nulo', lambda s: round(s.mean(), 1) if s.notna().any() else np.nan)
      ).sort_values('columnas', ascending=False).to_string())

print('\nPor que se excluye cada columna no elegible:')
print(D[D.elegible == 'no'].por_que_no.value_counts().to_string())

if 'pct_nulo' in D.columns:
    peor = D[D.pct_nulo.notna()].nlargest(12, 'pct_nulo')[['columna','familia','pct_nulo','elegible']]
    print('\nLas doce columnas con mas nulos:')
    print(peor.to_string(index=False))

## Paso 4 · Redundancia: cuánta información hay de verdad

Aquí está la pregunta de fondo del cuaderno. Las 141 variables **no son 141 piezas de
información**, y conviene saber cuántas son antes de afirmar que la base es rica.

Hay tres clases de redundancia y las tres se miden:

1. **Gemelos `_pct`.** Muchas variables vienen en pareja: la cantidad y su percentil dentro del
   estrato. Es una transformación monótona de la misma cosa, de modo que para un modelo de
   árboles son la misma variable vista dos veces.
2. **Banderas del CRI.** Las columnas `c_` y `k_` son umbrales de variables continuas que ya están
   en la base. Se detectan por una señal clara: tienen **exactamente la misma cobertura** que su
   variable de origen, porque se construyen a partir de ella.
3. **Colinealidad de rango.** Variables distintas de la misma familia que miden casi lo mismo.

De ahí sale la **dimensión efectiva**: cuántos bloques de información independiente hay a
|ρ| > 0,95. El paso 8 comprueba si usar un representante por bloque cuesta algo.


In [ ]:
# ============ Paso 4 · Redundancia: cuanta informacion hay de verdad ============
# La pregunta de fondo de este cuaderno: las 141 variables no son 141 piezas de informacion.
# Hay tres clases de redundancia y las tres se miden aqui.
#   1. Gemelos _pct: la misma cantidad y su percentil. Una transformacion monotona.
#   2. Banderas del CRI: son umbrales de una variable continua que ya esta en la base.
#   3. Colinealidad de rango entre variables distintas de la misma familia.
# Al final sale la DIMENSION EFECTIVA: cuantos bloques de informacion independiente hay.

N_MUESTRA_COR = 50000   # filas para la matriz de correlacion: 141x141 sobre todo el panel
UMBRAL        = 0.95    # |rho de Spearman| por encima del cual dos variables son la misma

PA = pd.read_parquet(f'{OUT}panel.parquet')
mod = [c for c in (VARS.get('A', []) + VARS.get('B', [])) if c in PA.columns]
mod = sorted(set(mod))
print(f'panel: {PA.shape[0]:,} filas | variables de modelo presentes: {len(mod)}')

# --- 1. Gemelos _pct ---
pares = [(c[:-4], c) for c in mod if c.endswith('_pct') and c[:-4] in PA.columns]
print(f'\n1. Gemelos _pct: {len(pares)} pares')
if pares:
    fil = []
    for a, b in pares:
        x, yv = PA[a], PA[b]
        m = x.notna() & yv.notna()
        rho = float(x[m].rank().corr(yv[m].rank())) if m.sum() > 100 else np.nan
        fil.append((a, b, round(rho, 4), int(m.sum()), a in mod, b in mod))
    G = pd.DataFrame(fil, columns=['base','gemelo_pct','rho_spearman','n','base_en_modelo','pct_en_modelo'])
    print(G.sort_values('rho_spearman', ascending=False).to_string(index=False))
    amb = G[(G.base_en_modelo) & (G.pct_en_modelo)]
    print(f'\n   pares donde AMBAS estan en el modelo: {len(amb)} de {len(G)}')
    print(f'   rho mediano entre gemelos: {G.rho_spearman.median():.4f}')
    if len(amb):
        print('   Para un modelo de arboles eso es informacion repetida: el arbol puede')
        print('   partir por cualquiera de las dos y el resultado es casi el mismo.')

# --- 2. Banderas del CRI como umbrales de una continua ---
cri = [c for c in PA.columns if c.startswith(('c_', 'k_'))]
print(f'\n2. Banderas del CRI en el panel: {len(cri)}')
if cri:
    fil = []
    for b in cri:
        nb = PA[b].isna().mean()
        cand = [c for c in PA.columns if c not in cri and not c.endswith('_pct')
                and abs(PA[c].isna().mean() - nb) < 1e-9]
        mejor, rmax = None, np.nan
        for c in cand:
            m = PA[b].notna() & PA[c].notna()
            if m.sum() < 100: continue
            r = abs(float(pd.to_numeric(PA[b][m], errors='coerce').corr(PA[c][m].rank())))
            if np.isnan(rmax) or r > rmax: mejor, rmax = c, r
        fil.append((b, round(nb*100, 2), mejor, round(rmax, 3) if rmax == rmax else np.nan))
    K = pd.DataFrame(fil, columns=['bandera','pct_nulo','misma_cobertura_que','|corr| con esa'])
    print(K.to_string(index=False))
    print('   Una bandera con la MISMA cobertura y correlacion alta con una continua es un')
    print('   umbral de esa continua. No agrega informacion, agrega una version gruesa.')

# --- 3. Colinealidad de rango y dimension efectiva ---
print(f'\n3. Colinealidad de rango entre las {len(mod)} variables de modelo')
sub = PA[mod].sample(min(N_MUESTRA_COR, len(PA)), random_state=SEMILLA)
Rk = sub.rank().to_numpy(dtype=float)
Rk = np.where(np.isnan(Rk), np.nanmean(Rk, axis=0), Rk)
Rk = Rk - Rk.mean(0); sd = Rk.std(0); sd[sd == 0] = 1.0
C  = np.abs((Rk / sd).T @ (Rk / sd) / len(Rk))
np.fill_diagonal(C, 0.0)

# componentes conexas con |rho| > UMBRAL
padre = list(range(len(mod)))
def raiz(i):
    while padre[i] != i: padre[i] = padre[padre[i]]; i = padre[i]
    return i
for i, j in zip(*np.where(np.triu(C > UMBRAL, 1))):
    ri, rj = raiz(int(i)), raiz(int(j))
    if ri != rj: padre[ri] = rj
grupos = {}
for i, c in enumerate(mod): grupos.setdefault(raiz(i), []).append(c)
bloques = sorted(grupos.values(), key=len, reverse=True)
print(f'   dimension efectiva a |rho| > {UMBRAL}: {len(bloques)} bloques de {len(mod)} variables')
print(f'   reduccion: {100*(1 - len(bloques)/len(mod)):.0f} %')
print(f'\n   Los bloques con mas de una variable:')
for b in [x for x in bloques if len(x) > 1][:12]:
    print(f'     {len(b)}: {b}')

REPRE = [b[0] for b in bloques]          # un representante por bloque
print(f'\n   conjunto reducido (un representante por bloque): {len(REPRE)} variables')

# --- varianza casi nula ---
nvar = [c for c in mod if PA[c].nunique(dropna=True) <= 2]
print(f'\n4. Variables con dos valores o menos: {len(nvar)}')
if nvar: print('   ', nvar[:20])

pd.DataFrame({'variable': mod,
              'bloque': [next(i for i, b in enumerate(bloques) if c in b) for c in mod],
              'es_representante': [c in REPRE for c in mod]}).to_csv(f'{OUTF}redundancia.csv', index=False)
print(f'\n  -> {OUTF}redundancia.csv')

## Paso 5 · Deriva temporal

Si una variable cambia de cobertura a lo largo de los años, el modelo puede estar aprendiendo el
**calendario** en lugar de la conducta. El percentil dentro del año protege contra cambios de
nivel, pero no contra cambios de cobertura: una variable que empieza a existir en 2019 divide la
muestra en dos épocas, y si los positivos no están repartidos igual entre ellas, el modelo tiene
un atajo.

El paso mide el rango de cobertura de cada variable entre años y marca las que se mueven más de
veinte puntos. También mira dos cortes concretos: la caída de actividad de 2019 a 2020, que es el
efecto de la pandemia, y 2026, que está incompleto hasta la fecha de corte.


In [ ]:
# ============ Paso 5 · Deriva temporal ============
# Si una variable cambia su cobertura o su nivel a lo largo de los anios, el modelo puede
# aprender el ANIO en lugar de la conducta. El percentil dentro del anio protege del nivel,
# pero no de un cambio de cobertura: eso hay que verlo.

anios = sorted(PA.anio_proceso.dropna().unique())
print(f'anios en el panel: {anios[0]} a {anios[-1]}')
print(PA.groupby('anio_proceso').size().rename('filas').to_frame().T.to_string())

# cobertura de cada variable por anio
cob = PA.groupby('anio_proceso')[mod].apply(lambda d: d.notna().mean() * 100)
rango = (cob.max() - cob.min()).sort_values(ascending=False)
print(f'\nVariables cuya COBERTURA se mueve mas de 20 puntos entre anios: '
      f'{int((rango > 20).sum())} de {len(mod)}')
if (rango > 20).any():
    t = pd.DataFrame({'rango_cobertura': rango[rango > 20].round(1),
                      'cob_min': cob.min()[rango > 20].round(1),
                      'cob_max': cob.max()[rango > 20].round(1),
                      'anio_min': cob.idxmin()[rango > 20],
                      'anio_max': cob.idxmax()[rango > 20]})
    print(t.head(15).to_string())
    print('   Un salto de cobertura es una ventana donde el dato empieza o deja de existir.')
    print('   Si coincide con los positivos, el modelo puede estar usando el calendario.')

# el corte 2019-2020
if 2019 in anios and 2020 in anios:
    a, b = PA[PA.anio_proceso == 2019], PA[PA.anio_proceso == 2020]
    print(f'\nCorte 2019 a 2020: {len(a):,} -> {len(b):,} filas '
          f'({100*(len(b)/len(a)-1):+.0f} %)')
    dif = (b[mod].notna().mean() - a[mod].notna().mean()) * 100
    peor = dif.abs().sort_values(ascending=False).head(8)
    print('   variables con mayor cambio de cobertura en ese corte:')
    print(pd.DataFrame({'cambio_pp': dif[peor.index].round(1)}).to_string())

# 2026 incompleto
if anios[-1] == 2026:
    m26 = PA[PA.anio_proceso == 2026]
    print(f'\n2026: {len(m26):,} filas. El anio esta incompleto hasta el corte del 5 de')
    print('   septiembre, asi que ningun NIVEL es comparable con otros anios. El percentil')
    print('   dentro del anio si lo es, y es la razon por la que la metrica es un percentil.')

# donde estan los positivos en el tiempo
for et in [c for c in ('y_fantasma', 'y_intragrupo') if c in PA.columns]:
    t = PA.groupby('anio_proceso')[et].sum().astype(int)
    print(f'\npositivos de {et} por anio: {dict(t)}')
    if t.sum(): print(f'   concentracion: el anio con mas positivos tiene '
                      f'{100*t.max()/t.sum():.0f} % del total')
cob.round(1).to_csv(f'{OUTF}cobertura_por_anio.csv')
print(f'\n  -> {OUTF}cobertura_por_anio.csv')

## Paso 6 · Universo: qué variables existen en cada procedimiento

El 83,5 % de los procesos son compras por catálogo electrónico, donde no hay competencia que
medir. Una variable que solo existe fuera del catálogo **no es una variable con nulos: es una
variable de otro universo**, y usarla sobre todo el panel mezcla dos poblaciones distintas.

Como el panel es proveedor por año y el procedimiento es por proceso, el paso calcula para cada
fila del panel qué fracción de sus procesos fue catálogo, y con eso clasifica a cada proveedor-año
en un perfil. Después compara la cobertura media de cada familia entre perfiles, y mira dónde
caen los positivos.


In [ ]:
# ============ Paso 6 · Universo: que variables existen en cada procedimiento ============
# El 83,5 % de los procesos son catalogo electronico y ahi no hay competencia que medir.
# Una variable que solo existe fuera del catalogo no es una variable con nulos: es una
# variable de OTRO universo. Esta tabla dice cual es cual.

try:
    pr = pd.read_parquet(f'{OUT9}procedimiento.parquet', columns=['ocid','clase','es_catalogo'])
    PV = pd.read_parquet(f'{OUTF}procesos.parquet', columns=['ocid','proveedor_ruc','anio_proceso'])
    pv = PV.merge(pr, on='ocid', how='left')
    # el panel es proveedor x anio: se le pega la fraccion de procesos de catalogo
    agg = pv.groupby(['proveedor_ruc','anio_proceso']).agg(
              procesos=('ocid','size'), pct_catalogo=('es_catalogo','mean')).reset_index()
    PB = PA.merge(agg, on=['proveedor_ruc','anio_proceso'], how='left')
    print(f'panel con mezcla de procedimiento: {PB.pct_catalogo.notna().mean()*100:.1f} % de filas cruzadas')
    PB['perfil'] = pd.cut(PB.pct_catalogo, [-0.01, 0.0, 0.5, 0.999, 1.0],
                          labels=['nada de catalogo','mas competitivo','mas catalogo','solo catalogo'])
    print('\nFilas del panel por perfil de procedimiento:')
    print(PB.perfil.value_counts(dropna=False).to_string())
    print('\nCobertura media de cada familia segun el perfil:')
    fams = {}
    for c in mod:
        k = c.split('_')[0] + '_' if '_' in c else 'base'
        fams.setdefault(k, []).append(c)
    fil = []
    for k, cs in sorted(fams.items()):
        r = {'familia': k, 'n_vars': len(cs)}
        for p in PB.perfil.cat.categories:
            s = PB[PB.perfil == p]
            r[str(p)] = round(s[cs].notna().mean().mean() * 100, 1) if len(s) else np.nan
        fil.append(r)
    print(pd.DataFrame(fil).to_string(index=False))
    print('\n   Una familia con cobertura alta solo en "nada de catalogo" pertenece al')
    print('   universo competitivo. Usarla sobre todo el panel mezcla dos poblaciones.')
    for et in [c for c in ('y_fantasma',) if c in PB.columns]:
        t = PB.groupby('perfil')[et].agg(['size','sum'])
        t['pct_positivos'] = (t['sum'] / max(1, t['sum'].sum()) * 100).round(1)
        print(f'\nDonde caen los positivos de {et}:')
        print(t.to_string())
except FileNotFoundError as e:
    print('falta un archivo para cruzar procedimiento con panel:', e)

## Paso 7 · Cuánta etiqueta falta

Esta es la parte de adelantarse a lo que puede pasar, y la conclusión es incómoda pero útil: **el
techo del trabajo no son las variables ni el modelo, son los 15 conglomerados positivos.**

El experimento submuestrea conglomerados positivos, de 4 hasta el total, y mide la amplitud del
intervalo del estadístico en cada tamaño. Se usa el **índice**, que no se entrena, para que lo que
se vea sea el efecto del tamaño de muestra y no el ruido del modelo.

La amplitud cae como una constante dividida por la raíz del número de conglomerados, lo que
permite responder la pregunta en la dirección útil: **cuántos conglomerados harían falta para
estrechar el intervalo a la mitad.** La respuesta es cuadrática, así que es mucho más de lo que
parece, y ninguna mejora del modelo la cambia.


In [ ]:
# ============ Paso 7 · Cuanta etiqueta falta: la curva de precision ============
# Esta es la parte de "adelantarnos a lo que pueda pasar". El techo del trabajo no son las
# 141 variables ni el modelo: son los 15 conglomerados positivos. Con 15, cualquier cifra
# trae una incertidumbre que hay que declarar, y ninguna prueba estadistica la arregla.
#
# El experimento es directo: se submuestrean conglomerados positivos, de 4 a 15, y se mide
# la amplitud del intervalo del estadistico. Se usa el INDICE, que no se entrena, para que
# lo que se vea sea el efecto del tamano de muestra y no el ruido del modelo.

COL_IDX = 'idx_intermediacion' if 'idx_intermediacion' in PA.columns else None
ET = 'y_fantasma'
if COL_IDX is None or ET not in PA.columns:
    print('falta idx_intermediacion o y_fantasma en el panel: no se puede hacer la curva')
else:
    pz_i = PA.groupby('anio_proceso')[COL_IDX].rank(pct=True).values
    rucs = PA.proveedor_ruc.astype(str).values
    yv   = PA[ET].values.astype(int)
    cl_pos = sorted(set(rucs[yv == 1]))
    print(f'conglomerados positivos: {len(cl_pos)} | filas positivas: {int(yv.sum())}')

    n_tot = len(cl_pos)
    m_tot = np.isin(rucs, cl_pos) & (yv == 1)
    print(f'valor con los {n_tot} conglomerados: {float(np.nanmedian(pz_i[m_tot])):.3f}'
          '  (un solo valor: no tiene intervalo porque no hay de donde submuestrear)')

    R_SUB = 300
    rr = np.random.default_rng(SEMILLA)
    fil = []
    for n_cl in [k for k in (4, 6, 8, 10, 12, 14, 20, 30) if k < n_tot]:
        vals = []
        for _ in range(R_SUB):
            sel = rr.choice(cl_pos, size=n_cl, replace=False)
            m = np.isin(rucs, sel) & (yv == 1)
            if m.sum() >= 3: vals.append(float(np.nanmedian(pz_i[m])))
        v = np.array(vals)
        if len(v) < 5:
            fil.append((n_cl, np.nan, np.nan, np.nan, np.nan, len(v))); continue
        lo, hi = np.percentile(v, [2.5, 97.5])
        fil.append((n_cl, round(float(v.mean()), 3), round(float(lo), 3), round(float(hi), 3),
                    round(float(hi - lo), 3), len(v)))
    CU = pd.DataFrame(fil, columns=['conglomerados','media','p2.5','p97.5','amplitud','n_submuestras'])
    print('\n' + CU.to_string(index=False))

    d = CU.dropna(subset=['amplitud'])
    d = d[d.n_submuestras > 5]
    if len(d) >= 3:
        # la amplitud de un intervalo cae como 1/raiz(n): se ajusta a = amplitud*raiz(n)
        a = float(np.median(d.amplitud * np.sqrt(d.conglomerados)))
        amp_hoy = a / np.sqrt(n_tot)
        print(f'\nLa amplitud del intervalo cae como {a:.2f} / raiz(conglomerados).')
        print(f'Con los {n_tot} de hoy la amplitud implicita es {amp_hoy:.3f}.')
        print('Para estrecharla hay que multiplicar los conglomerados, no afinar el modelo:')
        for f, et in [(0.5, 'la mitad'), (0.25, 'la cuarta parte')]:
            meta = amp_hoy * f
            print(f'  reducirla a {et} ({meta:.3f}) exige '
                  f'{int(np.ceil((a/meta)**2))} conglomerados positivos')
        print(f'\nHoy hay {n_tot}. Eso define la precision con la que se puede hablar,')
        print('y es un limite del DATO, no del metodo: ninguna mejora del modelo lo mueve.')
    CU.to_csv(f'{OUTF}curva_etiqueta.csv', index=False)
    print(f'\n  -> {OUTF}curva_etiqueta.csv')

## Paso 8 · Modelo de referencia

No es el modelo de la tesis y sus cifras **no son comparables** con el paso 7b del cuaderno 8:
aquí el protocolo es ligero —5 pliegues, 10 bolsas— porque la pregunta es otra. Lo que se
comprueba es dos cosas: que la base final sirve para modelar sin arreglos previos, y que el
conjunto reducido del paso 4 no pierde nada frente al completo.

Si la diferencia entre el conjunto completo y el reducido es pequeña, la base puede adelgazar sin
costo, y la tesis puede decir cuántas variables hacen falta de verdad. Si es grande, la
redundancia que midió el paso 4 no era redundancia y hay que revisar el umbral.


In [ ]:
# ============ Paso 8 · Modelo de referencia sobre la base final ============
# No es el modelo de la tesis: es una comprobacion de que la base final sirve para modelar y
# de que el conjunto reducido del paso 4 no pierde nada. Protocolo ligero y declarado:
# 5 pliegues y 10 bolsas. Las cifras NO son comparables con el paso 7b del cuaderno 8.
from sklearn.ensemble import HistGradientBoostingClassifier
K_REF, RATIO_U, N_POR_ANIO, N_PLIEGUES = 10, 5, 400, 5

def pu_bagging(Xp, Xu, Xs, k=K_REF, ratio=RATIO_U, seed=0):
    r = np.random.default_rng(seed); acum = np.zeros(len(Xs)); n = len(Xp); hechos = 0
    with contextlib.redirect_stderr(io.StringIO()):
        for i in range(k):
            idx = r.choice(len(Xu), size=min(len(Xu), n*ratio), replace=False)
            Xtr = np.vstack([Xp, Xu[idx]]); ytr = np.r_[np.ones(n), np.zeros(len(idx))]
            util = np.array([np.unique(Xtr[~np.isnan(Xtr[:,j]), j]).size >= 2
                             for j in range(Xtr.shape[1])])
            if util.sum() == 0: continue
            m = HistGradientBoostingClassifier(max_depth=3, max_iter=60, learning_rate=0.1,
                    min_samples_leaf=5, l2_regularization=1.0, random_state=i).fit(Xtr[:,util], ytr)
            acum += m.predict_proba(Xs[:,util])[:,1]; hechos += 1
    assert hechos > 0
    return acum / hechos

def evaluar(cols_u, etiqueta='y_fantasma', semilla=SEMILLA):
    d  = PA
    yv = d[etiqueta].values.astype(int)
    gi = pd.factorize(d.proveedor_ruc.astype(str).values)[0]
    an = d.anio_proceso.values
    X  = d[cols_u].astype(float).values
    gp = np.unique(gi[yv == 1])
    if len(gp) < 2: return np.nan
    r = np.random.default_rng(semilla)
    comp = []
    for a in np.unique(an):
        m = np.where((yv == 0) & (an == a))[0]
        if len(m): comp.append(r.choice(m, size=min(N_POR_ANIO, len(m)), replace=False))
    idx_s = np.concatenate([np.where(yv == 1)[0], np.concatenate(comp)])
    Xs, Xu = X[idx_s], X[yv == 0]
    sc = np.full(len(idx_s), np.nan); ac = np.zeros(len(idx_s)); nf = 0
    for j, fuera in enumerate([f for f in np.array_split(r.permutation(gp),
                               min(N_PLIEGUES, len(gp))) if len(f)]):
        dentro = np.isin(gi, fuera)
        s = pu_bagging(X[(yv == 1) & ~dentro], Xu, Xs, seed=semilla + j)
        don = dentro[idx_s] & (yv[idx_s] == 1); sc[don] = s[don]; ac += s; nf += 1
    falta = np.isnan(sc); sc[falta] = (ac / nf)[falta]
    ev = pd.DataFrame({'y': yv[idx_s], 'a': an[idx_s], 'm': sc})
    pct = ev.groupby('a')['m'].rank(pct=True).values
    return float(np.nanmedian(pct[ev.y.values == 1]))

casos = [('completo (todas las de modelo)', mod)]
if 'REPRE' in dir() and len(REPRE) < len(mod):
    casos.append((f'reducido (un representante por bloque)', REPRE))
sinpct = [c for c in mod if not c.endswith('_pct')]
if len(sinpct) < len(mod):
    casos.append(('sin gemelos _pct', sinpct))

fil = []
for nom, cs in casos:
    t0 = time.time()
    v = evaluar(cs)
    fil.append((nom, len(cs), round(v, 3) if v == v else np.nan, round(time.time() - t0)))
    print(f'  {nom:38s} {len(cs):3d} vars -> {v:.3f}   ({time.time()-t0:.0f} s)')
MR = pd.DataFrame(fil, columns=['conjunto','n_vars','estadistico','segundos'])
print('\n' + MR.to_string(index=False))
if len(MR) > 1 and MR.estadistico.notna().all():
    print(f'\nDiferencia entre el completo y el reducido: '
          f'{abs(MR.estadistico.iloc[0] - MR.estadistico.iloc[-1]):.3f}')
    print('Si es pequena, las variables de mas no aportan y la base puede adelgazar sin costo.')
MR.to_csv(f'{OUTF}modelo_referencia.csv', index=False)
print(f'  -> {OUTF}modelo_referencia.csv')

## Paso 9 · Las brechas

Lo que le falta a la base para ser la ideal, medido y con la fuente de donde saldría. Cada fila
sale de una cifra de este cuaderno, no de una opinión. La columna que más importa es la última:
de dónde se consigue lo que falta.


In [ ]:
# ============ Paso 9 · Que le falta a la base para ser la ideal ============
# Las carencias medidas, no opinadas. Cada fila sale de una cifra de este cuaderno.
bre = []
def brecha(q, cifra, impacto, fuente):
    bre.append({'que falta': q, 'cifra de hoy': cifra, 'que cambiaria': impacto, 'de donde sale': fuente})

n_cl = PA.loc[PA.y_fantasma == 1, 'proveedor_ruc'].nunique() if 'y_fantasma' in PA.columns else 0
brecha('Mas conglomerados etiquetados', f'{n_cl} RUC fantasma',
       'Es el techo de precision de todo el trabajo (paso 7)',
       'Actualizaciones del catastro del SRI; sanciones de la Contraloria; sentencias')
brecha('Oferentes, no solo adjudicatarios',
       f'n_oferentes nulo en el 83,5 % (estructural) y sin identidad de quien oferto',
       'Permitiria medir competencia real y no solo resultado',
       'Tabla de oferentes del portal por proceso')
ent_cols = [c for c in PA.columns if c.startswith('ent_')]
if ent_cols:
    cob_ent = 100 - PA[ent_cols].isna().mean().mean()*100
    brecha('Variables de la entidad contratante', f'cobertura media {cob_ent:.1f} %',
           ('La familia ent_ aporta 0,009 en la ablacion: con esta cobertura el limite no es'
            ' el dato sino que son pocas variables' if cob_ent > 50 else
            'La familia ent_ aporta 0,009 en la ablacion porque casi no tiene dato'),
           'Clasificador de entidades publicas; presupuesto por entidad')
brecha('Ejecucion del contrato', 'la base llega hasta la adjudicacion',
       'Sobreprecios, prorrogas y terminaciones anticipadas son senales de irregularidad',
       'Releases OCDS de etapa contract e implementation')
brecha('Beneficiario final', 'solo representante legal del directorio',
       'El objetivo B hoy usa representante legal como aproximacion de grupo',
       'Registro de beneficiarios finales; accionistas de la Superintendencia')
brecha('Dimension efectiva de las variables',
       f'{len(REPRE) if "REPRE" in dir() else "?"} bloques de {len(mod)} variables',
       'La base parece mas rica de lo que es: hay gemelos _pct y umbrales del CRI',
       'Paso 4 de este cuaderno')
B = pd.DataFrame(bre)
print(B.to_string(index=False))
B.to_csv(f'{OUTF}brechas.csv', index=False)
print(f'\n  -> {OUTF}brechas.csv')

## Paso 10 · Escribir y declarar

Dos archivos autoritativos, uno por grano, con su huella. Siete archivos de acompañamiento. Y una
lista de candidatos a `09_historico/` que hay que **revisar antes de mover nada**: se arma por
nombre de archivo, no por uso real, así que puede incluir algo que todavía haga falta.


In [ ]:
# ============ Paso 10 · Escribir el panel final y declarar lo historico ============
# El panel es el segundo grano del trabajo y no se puede meter en la misma tabla que los
# procesos: uno es una fila por proceso y el otro una fila por proveedor y anio. "Una sola
# base" significa UNA FUENTE DE VERDAD por grano, no una sola tabla imposible.
nA = len(PA)
PF = PA.copy()
for rol, llave, pref in [('supercias', 'proveedor_ruc', ('sup_', 'rep_', 'doc_')),
                         ('indices', None, ('idx_',))]:
    ruta = FUENTES.get(rol, '')
    if not os.path.exists(ruta): print(f'  {rol}: ausente, no se une'); continue
    T = pd.read_parquet(ruta)
    if rol == 'indices':
        k = [c for c in ('proveedor_ruc', 'anio') if c in T.columns]
        if len(k) == 2:
            T = T.rename(columns={'anio': 'anio_proceso'})
            k = ['proveedor_ruc', 'anio_proceso']
        else: print('  indices: sin llave proveedor_ruc + anio'); continue
    else:
        k = [llave]
    nuevas = [c for c in T.columns if c.startswith(pref) and c not in PF.columns]
    if not nuevas: print(f'  {rol}: nada nuevo que unir'); continue
    T = T[k + nuevas].drop_duplicates(subset=k)
    PF = PF.merge(T, on=k, how='left')
    print(f'  {rol}: +{len(nuevas)} columnas, filas {nA:,} -> {len(PF):,}'
          + ('  OK' if len(PF) == nA else '  ALTO: la union multiplico filas'))

assert len(PF) == nA, 'una union multiplico filas del panel'
PF.to_parquet(f'{OUTF}panel.tmp.parquet', index=False)
os.replace(f'{OUTF}panel.tmp.parquet', f'{OUTF}panel.parquet')
print(f'\npanel final: {PF.shape[0]:,} filas x {PF.shape[1]} columnas')
print(f'  -> {OUTF}panel.parquet')

print('\n' + '='*74); print('LA BASE FINAL, EN DOS ARCHIVOS'); print('='*74)
res = []
for nom, ruta in [('procesos (una fila por ocid)', f'{OUTF}procesos.parquet'),
                  ('panel (una fila por proveedor y anio)', f'{OUTF}panel.parquet')]:
    if os.path.exists(ruta):
        res.append((nom, os.path.basename(ruta), filas(ruta), len(cols(ruta)),
                    round(os.path.getsize(ruta)/1e6, 1),
                    hashlib.sha256(open(ruta, 'rb').read(1 << 20)).hexdigest()[:16]))
print(pd.DataFrame(res, columns=['grano','archivo','filas','columnas','mb','huella_1mb']).to_string(index=False))

print('\nAcompanan a la base:')
for a in ['diccionario.csv', 'pruebas.csv', 'redundancia.csv', 'cobertura_por_anio.csv',
          'curva_etiqueta.csv', 'modelo_referencia.csv', 'brechas.csv']:
    r = f'{OUTF}{a}'
    print(f'  {"si" if os.path.exists(r) else "NO"}  final_{a}')

if MAN is not None:
    hist = MAN[MAN.clase_archivo == '3 historico']
    if len(hist):
        with open(f'{OUTF}mover_a_historico.txt', 'w', encoding='utf-8') as f:
            f.write('# archivos que ya no alimentan nada: mover a 09_historico/\n')
            for a in hist.archivo: f.write(str(a) + '\n')
        print(f'\n  -> {OUTF}mover_a_historico.txt  ({len(hist)} archivos, '
              f'{hist.mb.sum()/1024:.2f} GB)')
        print('  Revisalo antes de mover nada: la lista se arma por nombre, no por uso.')

## Cierre · Qué queda, en una página

Después de correr este cuaderno hay dos archivos que son la base, siete que la documentan, y tres
respuestas que antes no se tenían.

**La base.** `final_procesos.parquet`, una fila por proceso, y `final_panel.parquet`, una fila por
proveedor y año. Todo lo demás es derivado o histórico, y la lista de lo histórico está escrita.

**Las tres respuestas.**

1. *¿Cuánta información hay?* La dimensión efectiva del paso 4, que es menor que el número de
   variables. Si el paso 8 confirma que el conjunto reducido empata con el completo, la tesis
   puede reportar el número honesto de variables en lugar del número grande.
2. *¿Alcanza la etiqueta?* La curva del paso 7 dice la amplitud del intervalo con los
   conglomerados de hoy y cuántos harían falta para estrecharla. Es un límite del dato, no del
   método, y es el argumento más fuerte para la sección de trabajo futuro.
3. *¿Qué conseguir?* Las brechas del paso 9, cada una con la fuente de donde saldría.

**Lo que este cuaderno no hace, y conviene decirlo.** No valida el modelo —eso es el cuaderno 8—
ni responde la circularidad —eso es el 11— ni registra las fallas —eso es el 10—. Hace una sola
cosa: dejar la base en un estado en el que alguien de fuera pueda recogerla, entenderla y
confiar en ella sin preguntar nada.


In [ ]:
# ========================================================================
# Paso 11 - El catalogo de bases para los companeros
# Va AL FINAL, despues del paso 10. Es lo ultimo que corre.
# Copia TODO este archivo y pegalo en una celda nueva del cuaderno 12.
# ========================================================================

# ============ Paso 11 · El catalogo de bases para los companeros ============
# Lo que Santiago y Andres necesitan: que tablas quedan, para que sirve cada una, cual es su
# llave, cuantas filas tiene y que significa cada variable. Sale en Excel, una hoja por tabla,
# y en Markdown para pegar en el documento de la tesis.

DESC_PREF = {
 'red_': 'Red bipartita entidad-proveedor',
 'cob_': 'Red de co-oferta entre proveedores (rivales)',
 'com_': 'Comunidades detectadas en la red de co-oferta',
 'ent_': 'Entidad contratante',
 'sup_': 'Directorio societario de la Superintendencia de Companias',
 'rep_': 'Representante legal',
 'idx_': 'Indice teorico construido en la capa 2',
 'cri2': 'Indice tipo CRI, version 2',
 'c_':   'Bandera roja binaria del indice tipo CRI',
 'k_':   'Bandera roja binaria del indice tipo CRI',
 'f_':   'Fecha de la marca del SRI',
 'y_':   'Etiqueta de validacion',
 'doc_': 'Estado documental de la compania',
 'dim_': 'Dimension descriptiva del proveedor-anio',
}
DESC_RAIZ = {
 'ocid': 'identificador OCDS del proceso; es la llave del grano proceso',
 'proveedor_ruc': 'RUC del proveedor adjudicatario; parte de la llave del panel',
 'entidad_ruc': 'RUC de la entidad contratante',
 'anio_proceso': 'anio del proceso; parte de la llave del panel',
 'fecha_release': 'fecha del release OCDS del que sale la fila',
 'monto_final': 'monto del contrato; es la columna de monto canonica del trabajo',
 'monto_anio': 'monto contratado por el proveedor en el anio',
 'monto_max': 'monto del contrato mas grande',
 'monto_total': 'monto acumulado',
 'monto_tender': 'monto referencial del pliego; EXCLUIDO del modelo por su origen',
 'monto_award': 'monto adjudicado',
 'n_entidades': 'numero de entidades contratantes distintas',
 'n_contratos': 'numero de contratos',
 'n_procesos': 'numero de procesos',
 'n_rivales': 'numero de proveedores distintos que compitieron en los mismos procesos',
 'n_oferentes': 'numero de oferentes del proceso; nulo donde no hay proceso de ofertas',
 'n_parties': 'numero de partes declaradas en el release OCDS',
 'n_comunidades': 'numero de comunidades distintas entre los rivales',
 'n_comp_': 'cuantos componentes del indice estaban disponibles en esa fila',
 'autovector': 'centralidad de autovector: importancia por la importancia de sus vecinos',
 'coreness': 'profundidad en el nucleo de la red (k-core)',
 'hhi': 'indice de Herfindahl: 0 reparto total, 1 un solo actor',
 'agrupamiento': 'coeficiente de agrupamiento: que tan cerrado es el vecindario',
 'cuota_top': 'cuota de la contraparte principal',
 'cuota_rival_top': 'cuota del rival principal',
 'captura_max': 'captura maxima por una sola contraparte',
 'captura_media': 'captura promedio',
 'rivales_recurrentes': 'rivales que reaparecen en mas de un proceso',
 'rivales_x_proceso': 'rivales por proceso',
 'peso_rivales': 'peso agregado de los rivales',
 'participacion': 'participacion dentro de la comunidad',
 'z_intra': 'puntaje z de la interaccion dentro de la propia comunidad',
 'tamano': 'tamano',
 'win_rate': 'tasa de adjudicacion sobre los procesos en que participo',
 'dominancia': 'dominancia maxima historica frente a una entidad',
 'concentracion_rival': 'concentracion del rival habitual',
 'ratio_monto_capital': 'razon entre monto contratado y capital suscrito',
 'capital_suscrito': 'capital suscrito declarado',
 'anios_desde': 'anios transcurridos desde el hito que nombra la columna',
 'tipo_contratacion': 'descripcion original del tipo de contratacion (1.644 valores distintos)',
 'familia_procedimiento': 'familia de procedimiento normalizada',
 'clase': 'clase de procedimiento normalizada en cinco categorias',
 'es_catalogo': 'verdadero si el proceso es compra por catalogo electronico',
 'universo_competencia': 'verdadero si el proceso pertenece al universo de competencia',
 'proveedor_unico': 'verdadero si hubo un solo oferente; NULO donde no hay proceso de ofertas',
 'y_fantasma': 'etiqueta de validacion: el proveedor figura en el catastro de empresas fantasma del SRI',
 'y_intragrupo': 'etiqueta de validacion: procesos entre empresas del mismo representante legal',
 'tiene_grupo': 'verdadero si el proveedor tiene grupo societario identificado',
 'aparece_oferentes': 'verdadero si el proveedor aparece en la tabla de oferentes',
 'en_catastro': 'verdadero si el RUC aparece en el catastro del SRI',
 'provincia_entidad': 'provincia de la entidad contratante',
 'cpc5_principal': 'clasificacion CPC a cinco digitos del objeto principal',
 'rep_norm': 'nombre del representante legal normalizado; es la llave del grano grupo',
 'anonima': 'indicador: sociedad anonima',
 'responsabilidad_limi': 'indicador: sociedad de responsabilidad limitada',
 'sociedad_por_accione': 'indicador: sociedad por acciones simplificada',
 'sucursal': 'indicador: sucursal de compania extranjera',
 'asociacion_o_consorc': 'indicador: asociacion o consorcio',
 'anio_constitucion': 'anio de constitucion de la compania',
 'provincia': 'provincia registrada',
 'tipo': 'tipo societario declarado',
 'proveedor_nombre': 'razon social del proveedor',
 'tiene_periodo_competitivo': 'verdadero si el proceso tuvo periodo de recepcion de ofertas',
 'es_nuevo_en_mercado': 'indicador: es el primer anio de actividad del proveedor',
 'fue_suspendido': 'indicador: el proveedor estuvo suspendido en algun momento',
 'fecha_suspension': 'fecha de suspension definitiva registrada',
 'es_sociedad_registrada': 'indicador: el RUC aparece en el directorio de companias',
 'es_fantasma': 'indicador: el RUC figura en el catastro de empresas fantasma del SRI',
 'fantasma_estado': 'estado de la marca del SRI; derivado de la etiqueta, EXCLUIDO del modelo',
 'fantasma_detalle': 'detalle de la marca del SRI; derivado de la etiqueta, EXCLUIDO del modelo',
 'intermediacion': 'indice de intermediacion: ocho componentes de red, signos de Burt y Guimera-Amaral',
 'cartel': 'indice de cartel: seis componentes de red, signos de Wachs-Kertesz',
 'n_proveedores': 'numero de proveedores distintos',
 'fecha_proc': 'fecha del proceso',
 'f_marca': 'fecha en que el SRI marco al proveedor como fantasma',
 'f_reactivacion': 'fecha de reactivacion del RUC',
 'momento': 'posicion del proceso respecto de la fecha de la marca: antes, durante o despues',
 'no_activa': 'indicador: la compania no esta activa en el directorio',
 'grupo_tamano': 'estrato de tamano del proveedor',
 'oferente_ruc': 'RUC de un oferente del proceso, haya ganado o no',
 'fecha_fin_tender': 'fecha de cierre de la recepcion de ofertas',
 'nombre': 'razon social',
 'situacion legal': 'situacion legal registrada en la Superintendencia',
 'fecha_constitucion': 'fecha de constitucion de la compania',
 'representante': 'nombre del representante legal como consta en el directorio',
 'cargo': 'cargo del representante',
 'ruc': 'RUC',
 'anio': 'anio',
 'rep_tasa_intragrupo': 'proporcion de procesos del representante que son entre sus propias empresas',
 'rep_proc_intragrupo': 'numero de procesos del representante entre sus propias empresas',
 'com_id': 'identificador de la comunidad asignada al proveedor en la red de co-oferta',
 'c_unico': 'bandera roja: el proceso tuvo oferente unico',
 'c_conc_rival': 'bandera roja: la concentracion del rival habitual supera el umbral',
 'c_winrate': 'bandera roja: la tasa de adjudicacion supera el umbral',
 'c_dominancia': 'bandera roja: la dominancia frente a una entidad supera el umbral',
 'k_ratio_capital': 'bandera roja: la razon monto sobre capital supera el umbral',
 'k_cap_minimo': 'bandera roja: el capital suscrito esta en el minimo legal',
 'k_nuevo': 'bandera roja: el proveedor es nuevo en el mercado',
 'cri2_medio': 'indice CRI v2: promedio de las banderas rojas disponibles',
 'cri2_max': 'indice CRI v2: valor maximo entre las banderas rojas',
 'dim_competencia': 'dimension agregada de competencia del proveedor-anio',
 'dim_capacidad': 'dimension agregada de capacidad del proveedor-anio',
 'dim_montos': 'dimension agregada de montos del proveedor-anio',
 'situacion legal': 'situacion legal registrada en la Superintendencia',
 'capital suscrito': 'capital suscrito declarado en el directorio',
}
PROVINCIAS = ('azuay','bolivar','canar','carchi','chimborazo','cotopaxi','el_oro','esmeraldas',
 'galapagos','guayas','imbabura','loja','los_rios','manabi','morona','napo','orellana','pastaza',
 'pichincha','santa_elena','santo_domingo','sucumbios','tungurahua','zamora')
PENDIENTE = '[FALTA DESCRIPCION: escribirla a mano]'

import unicodedata
def _sin_tildes(x):
    return ''.join(ch for ch in unicodedata.normalize('NFD', x)
                   if unicodedata.category(ch) != 'Mn')

def describe(c):
    cl  = _sin_tildes(c.lower())
    pct = cl.endswith('_pct')
    base = cl[:-4] if pct else cl
    fam  = next((v for k, v in DESC_PREF.items() if cl.startswith(k)), None)
    pv   = next((p for p in PROVINCIAS if base.endswith('_' + p)), None)
    if pv and fam:
        return f'{fam}: indicador de que la compania esta registrada en {pv.replace("_", " ").title()}'
    raiz = None
    for k in sorted(DESC_RAIZ, key=len, reverse=True):
        if k in base: raiz = DESC_RAIZ[k]; break
    suf = ' (percentil dentro del estrato anio x decil de tamano)' if pct else ''
    if raiz and fam:  return f'{fam}: {raiz}{suf}'
    if raiz:          return raiz[0].upper() + raiz[1:] + suf
    if fam:           return f'{fam}: {PENDIENTE}{suf}'
    return PENDIENTE

# --- las tablas del catalogo, en los cuatro niveles ---
TABLAS = [
 (1, f'{OUTF}procesos.parquet', 'ocid', 'Una fila por proceso de contratacion. Es la base de la que sale todo lo demas.'),
 (1, f'{OUTF}panel.parquet', 'proveedor_ruc + anio_proceso', 'Una fila por proveedor y anio. Es la tabla sobre la que corren los modelos.'),
 (2, f'{OUT9}procedimiento.parquet', 'ocid', 'Taxonomia: normaliza los 1.644 valores de tipo_contratacion a cinco clases.'),
 (2, f'{OUT}etiqueta.parquet', 'ocid', 'Etiquetas de validacion por proceso, con la fecha de la marca del SRI.'),
 (2, f'{OUT}supercias.parquet', 'proveedor_ruc', 'Directorio societario: tipo, provincia, representante legal.'),
 (2, f'{R}/directorio_companias.parquet', 'ruc', 'Directorio completo de la Superintendencia de Companias.'),
 (2, f'{R}/oferentes_consolidado.parquet', 'ocid + oferente', 'Quien oferto en cada proceso, no solo quien gano.'),
 (3, f'{R}/indices_red.parquet', 'proveedor_ruc + anio', 'Indices teoricos de intermediacion y cartel. Derivado: se puede regenerar.'),
 (3, f'{OUT}cri_proceso.parquet', 'ocid', 'Banderas rojas tipo CRI por proceso. Derivado.'),
 (3, f'{OUT}scores.parquet', 'proveedor_ruc + anio', 'Puntajes del modelo. Derivado.'),
]

idx, hojas = [], {}
for nivel, ruta, llave, prop in TABLAS:
    if not os.path.exists(ruta):
        idx.append({'nivel': nivel, 'tabla': os.path.basename(ruta), 'estado': 'no existe',
                    'llave': llave, 'filas': None, 'columnas': None, 'mb': None, 'proposito': prop})
        continue
    cs, nf = cols(ruta), filas(ruta)
    idx.append({'nivel': nivel, 'tabla': os.path.basename(ruta), 'estado': 'presente',
                'llave': llave, 'filas': nf, 'columnas': len(cs),
                'mb': round(os.path.getsize(ruta)/1e6, 1), 'proposito': prop})
    if nivel <= 2:
        tip = {r[0]: r[1] for r in con.sql(
            f"DESCRIBE SELECT * FROM read_parquet('{ruta}')").fetchall()}
        nul = seguro(lambda: con.sql(f"""SELECT * FROM (SUMMARIZE
            SELECT * FROM read_parquet('{ruta}'))""").df())
        mapa_nul = {}
        if nul is not None and 'column_name' in nul.columns:
            mapa_nul = dict(zip(nul.column_name, nul.null_percentage))
        hojas[os.path.basename(ruta).replace('.parquet', '')[:31]] = pd.DataFrame([{
            'columna': c, 'tipo': t,
            'pct_nulo': (round(float(mapa_nul.get(c) or 0), 2) if mapa_nul else None),
            'es_llave': c in [x.strip() for x in llave.split('+')],
            'descripcion': describe(c)} for c, t in tip.items()])

IDX = pd.DataFrame(idx)
print('='*74); print('CATALOGO DE BASES'); print('='*74)
print(IDX[['nivel','tabla','estado','llave','filas','columnas','mb']].to_string(index=False))

nivel_txt = {1: 'LA BASE. Se modela sobre estas.',
             2: 'REFERENCIA. Se consultan para construir, no se modelan.',
             3: 'DERIVADAS. Se pueden borrar y regenerar.',
             4: 'HISTORICO. Evidencia de trazabilidad, no alimentan nada.'}
for n in sorted(IDX.nivel.unique()):
    print(f'\nNivel {n} — {nivel_txt.get(n, "")}')
    for _, r in IDX[IDX.nivel == n].iterrows():
        est = '' if r.estado == 'presente' else '  (NO EXISTE)'
        nf_ = '' if pd.isna(r.filas) else format(int(r.filas), ',') + ' filas'
        print(f'  {r.tabla:34s} llave: {r.llave:30s} {nf_}{est}')

pend = sum(int((h.descripcion.str.contains('FALTA DESCRIPCION')).sum()) for h in hojas.values())
tot  = sum(len(h) for h in hojas.values())
print(f'\nDescripciones: {tot - pend} de {tot} generadas automaticamente | '
      f'{pend} por escribir a mano')
if pend:
    print('Las que faltan, por tabla:')
    for k, h in hojas.items():
        f_ = h[h.descripcion.str.contains('FALTA DESCRIPCION')]
        if len(f_): print(f'  {k}: {len(f_)}  ->  {list(f_.columna)[:8]}')

# --- Excel, una hoja por tabla ---
xls = f'{OUTF}catalogo.xlsx'
try:
    with pd.ExcelWriter(xls, engine='openpyxl') as w:
        IDX.to_excel(w, sheet_name='indice', index=False)
        for k, h in hojas.items(): h.to_excel(w, sheet_name=k, index=False)
    print(f'\n  -> {xls}  ({1 + len(hojas)} hojas)')
except Exception as ex:
    print(f'\nno pude escribir el Excel ({type(ex).__name__}). Instala openpyxl con:')
    print('   pip install openpyxl')
    IDX.to_csv(f'{OUTF}catalogo_indice.csv', index=False)
    for k, h in hojas.items(): h.to_csv(f'{OUTF}catalogo_{k}.csv', index=False)
    print(f'  -> {OUTF}catalogo_indice.csv y {len(hojas)} CSV por tabla')

# --- Markdown para pegar en la tesis ---
with open(f'{OUTF}catalogo.md', 'w', encoding='utf-8') as f:
    f.write('# Catalogo de bases de datos del trabajo\n\n')
    f.write(f'Generado el {datetime.datetime.now():%d de %B de %Y}. '
            'Cada tabla con su llave, su grano y su proposito.\n\n')
    for n in sorted(IDX.nivel.unique()):
        f.write(f'## Nivel {n} — {nivel_txt.get(n, "")}\n\n')
        f.write('| Tabla | Llave | Filas | Columnas | Para que sirve |\n|---|---|---|---|---|\n')
        for _, r in IDX[IDX.nivel == n].iterrows():
            fl = '—' if pd.isna(r.filas) else f'{int(r.filas):,}'
            nc = '—' if pd.isna(r.columnas) else str(int(r.columnas))
            f.write(f'| `{r.tabla}` | `{r.llave}` | {fl} | {nc} | {r.proposito} |\n')
        f.write('\n')
    for k, h in hojas.items():
        f.write(f'## Variables de `{k}`\n\n')
        f.write('| Columna | Tipo | % nulo | Llave | Significado |\n|---|---|---|---|---|\n')
        for _, r in h.iterrows():
            f.write(f'| `{r.columna}` | {r.tipo} | '
                    f'{"—" if r.pct_nulo is None else f"{r.pct_nulo:.1f}"} | '
                    f'{"si" if r.es_llave else ""} | {r.descripcion} |\n')
        f.write('\n')
print(f'  -> {OUTF}catalogo.md  (para pegar en el documento)')